In [ ]:
%pip install -q scikit-learn pandas numpy matplotlib nltk pymorphy3 razdel
%pip install -q spacy && python -m spacy download en_core_web_sm

In [3]:
import re
import time
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", 120)
print("Окружение готово")

Matplotlib is building the font cache; this may take a moment.


Окружение готово


In [6]:
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
HTML_RE    = re.compile(r"<[^>]+>")
MENTION_RE = re.compile(r"[@#]\w+")
DIGIT_RE   = re.compile(r"\d+([.,]\d+)?")
SPACE_RE   = re.compile(r"\s+")
EMOJI_RE   = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF" "\U0001F1E0-\U0001F1FF" "]",
    flags=re.UNICODE,
)


def clean_text(text: str,
               keep_emoji: bool = True,
               mask_numbers: bool = True,
               fix_yo: bool = True) -> str:
    '''Базовая нормализация сырого текста.

    keep_emoji   — эмодзи выносим в отдельные токены (для тональности они полезны)
    mask_numbers — все числа схлопываем в один токен NUM (иначе словарь взорвётся)
    fix_yo       — ё → е, иначе «ещё» и «еще» будут разными признаками
    '''
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" ", text)

    if keep_emoji:
        text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    else:
        text = EMOJI_RE.sub(" ", text)

    text = text.lower()
    if fix_yo:
        text = text.replace("ё", "е")
    if mask_numbers:
        text = DIGIT_RE.sub(" NUM ", text)

    # оставляем буквы, пробелы и эмодзи; пунктуацию убираем
    text = re.sub(r"[^a-zA-Zа-яА-Я\s\U0001F300-\U0001FAFF\U00002600-\U000027BF]", " ", text)
    return SPACE_RE.sub(" ", text).strip()


print(clean_text(raw))

очень плохое качество заказывал тут url пришло за NUM дней размер xl не подошел шов кривой 😡 😡 цена NUM NUM руб не берите мой отзыв NUM NUM


In [10]:
sample = "Не понравилось и: размер XL, пришёл за 10 дней(( Санкт-Петербург, что-то не то. Цена 1 999 руб."

# 1) наивный split
by_split = sample.split()

# 2) регулярка \w+
by_regex = re.findall(r"\w+", sample)

# 3) razdel — правила, заточенные под русский
try:
    from razdel import tokenize as razdel_tokenize
    by_razdel = [t.text for t in razdel_tokenize(sample)]
except ImportError:
    by_razdel = ["<razdel не установлен>"]

# 4) nltk (универсальный, но обучен на английском)
try:
    import nltk
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    by_nltk = nltk.word_tokenize(sample, language="russian")
except Exception as e:
    by_nltk = [f"<nltk недоступен: {e}>"]

for name, toks in [("split", by_split), ("regex", by_regex),
                   ("razdel", by_razdel), ("nltk", by_nltk)]:
    print(f"{name:8s} ({len(toks):2d}): {toks}")

split    (17): ['Не', 'понравилось', 'и:', 'размер', 'XL,', 'пришёл', 'за', '10', 'дней((', 'Санкт-Петербург,', 'что-то', 'не', 'то.', 'Цена', '1', '999', 'руб.']
regex    (19): ['Не', 'понравилось', 'и', 'размер', 'XL', 'пришёл', 'за', '10', 'дней', 'Санкт', 'Петербург', 'что', 'то', 'не', 'то', 'Цена', '1', '999', 'руб']
razdel   (24): ['Не', 'понравилось', 'и', ':', 'размер', 'XL', ',', 'пришёл', 'за', '10', 'дней', '(', '(', 'Санкт-Петербург', ',', 'что-то', 'не', 'то', '.', 'Цена', '1', '999', 'руб', '.']
nltk     (24): ['Не', 'понравилось', 'и', ':', 'размер', 'XL', ',', 'пришёл', 'за', '10', 'дней', '(', '(', 'Санкт-Петербург', ',', 'что-то', 'не', 'то', '.', 'Цена', '1', '999', 'руб', '.']


In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("russian")

try:
    import pymorphy3
    morph = pymorphy3.MorphAnalyzer()
    HAS_MORPH = True
except ImportError:
    HAS_MORPH = False
    print("pymorphy3 не установлен — лемматизация будет пропущена")

words = "стали покупать качественные вещи в магазинах города Екатеринбурга".split()

rows = []
for w in words:
    lemma = morph.parse(w)[0].normal_form if HAS_MORPH else "—"
    pos = str(morph.parse(w)[0].tag.POS) if HAS_MORPH else "—"
    rows.append({"слово": w, "стем": stemmer.stem(w), "лемма": lemma, "часть речи": pos})

pd.DataFrame(rows)

In [ ]:
if HAS_MORPH:
    for p in morph.parse("стали")[:4]:
        print(f"{p.normal_form:10s} {str(p.tag):45s} score={p.score:.3f}")

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=200_000)
def lemmatize_word(word: str) -> str:
    return morph.parse(word)[0].normal_form if HAS_MORPH else word


def lemmatize_text(text: str) -> str:
    return " ".join(lemmatize_word(w) for w in text.split())


def stem_text(text: str) -> str:
    return " ".join(stemmer.stem(w) for w in text.split())


demo_sentence = "стали покупать качественные вещи в магазинах Екатеринбурга"
print("стемы:", stem_text(demo_sentence))
print("леммы:", lemmatize_text(demo_sentence))

In [ ]:
RUREVIEWS_URL = (
    "https://raw.githubusercontent.com/sismetanin/rureviews/master/"
    "women-clothing-accessories.3-class.balanced.csv"
)

def load_data() -> pd.DataFrame:
    try:
        df = pd.read_csv(RUREVIEWS_URL, sep="\t")
        print(f"Загружен RuReviews: {df.shape[0]} строк")
    except Exception as e:
        print(f"Не удалось скачать ({e}). Использую синтетический fallback.")
        pos = ["отличное качество, всем советую", "пришло быстро, размер идеальный",
               "супер вещь, довольна покупкой", "качество на высоте, спасибо продавцу"]
        neg = ["ужасное качество, ткань рвется", "не пришло, деньги не вернули",
               "размер не соответствует, брак", "кошмар, не советую никому"]
        neu = ["пришло вовремя, ничего особенного", "нормально, по цене соответствует",
               "обычная вещь, без восторга", "заказ получен, претензий нет"]
        rows = []
        for lbl, base in [("positive", pos), ("negative", neg), ("neautral", neu)]:
            for i in range(1500):
                rows.append({"review": random.choice(base) + " " * (i % 2), "sentiment": lbl})
        df = pd.DataFrame(rows)
    return df


data = load_data()
# в исходных метках есть опечатка: neautral вместо neutral. Реальные данные всегда грязные.
data["sentiment"] = data["sentiment"].replace({"neautral": "neutral"})
data.head()

In [ ]:
print("Распределение классов:")
print(data["sentiment"].value_counts(), "\n")

data["n_chars"] = data["review"].str.len()
data["n_words"] = data["review"].str.split().str.len()
print(data.groupby("sentiment")[["n_chars", "n_words"]].median())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
data["n_words"].clip(upper=100).hist(bins=50, ax=ax[0])
ax[0].set_title("Длина отзыва (слов)")
data["sentiment"].value_counts().plot.bar(ax=ax[1], rot=0)
ax[1].set_title("Баланс классов")
plt.tight_layout(); plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["review"], data["sentiment"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=data["sentiment"],
)
print(f"train: {len(X_train)}   test: {len(X_test)}")

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3)),
    ("clf",   LogisticRegression(max_iter=1000, C=5, random_state=RANDOM_STATE)),
])

t0 = time.perf_counter()
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)
print(f"обучение: {time.perf_counter() - t0:.1f} с\n")
print(classification_report(y_test, pred, digits=3))

In [47]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(pipeline, X=X_train, y=y_train, cv=cv_strategy) -> tuple:
    '''Честная кросс-валидация: fit векторизатора происходит внутри каждого фолда.'''
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean(), scores.std()

SUB = 20_000
X_sub, y_sub = X_train[:SUB], y_train[:SUB]

mean, std = evaluate(baseline, X_sub, y_sub)
print(f"baseline macro-F1 = {mean:.4f} ± {std:.4f}")

baseline macro-F1 = 0.7200 ± 0.0072


---
## 8. Сетка экспериментов

Теперь проверяем гипотезы, а не верим слайдам. Каждая строка ниже — отдельное утверждение,
которое мы либо подтверждаем, либо опровергаем числом.

In [ ]:
experiments = {
    "1. word 1-грамм":
        TfidfVectorizer(min_df=3),
    "2. word 1-2 грамм (baseline)":
        TfidfVectorizer(ngram_range=(1, 2), min_df=3),
    "3. word 1-2 + БЕЗ стоп-слов":
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, stop_words=list(RU_STOPWORDS)),
    "4. word 1-2 + sublinear_tf":
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True), # вместо tf используется 1 + log(tf)
    "5. BoW вместо TF-IDF":
        CountVectorizer(ngram_range=(1, 2), min_df=3),
    "6. char_wb 3-5 грамм":
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3),
    "7. union: word + char":
        FeatureUnion([
            ("w", TfidfVectorizer(ngram_range=(1, 2), min_df=3)),
            ("c", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3)),
        ]),
}

results = []
for name, vec in experiments.items():
    pipe = Pipeline([("vec", vec),
                     ("clf", LogisticRegression(max_iter=1000, C=5, random_state=RANDOM_STATE))])
    t0 = time.perf_counter()
    mean, std = evaluate(pipe, X_sub, y_sub)
    n_feat = len(pipe.fit(X_sub, y_sub).named_steps["vec"].get_feature_names_out())
    results.append({"конфигурация": name, "macro-F1": round(mean, 4),
                    "±": round(std, 4), "признаков": n_feat,
                    "время, с": round(time.perf_counter() - t0, 1)})
    print(f"{name:32s} F1={mean:.4f}")

res_df = pd.DataFrame(results).sort_values("macro-F1", ascending=False).reset_index(drop=True)
res_df

---
## 9. Интерпретация: что именно выучила модель

Огромное преимущество классики над нейросетями — модель можно **прочитать**.
Веса логистической регрессии на TF-IDF напрямую говорят, какие слова тянут к какому классу.

In [ ]:
best = Pipeline([
    ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=5)),
    ("clf", LogisticRegression(max_iter=1000, C=5, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

feature_names = np.array(best.named_steps["vec"].get_feature_names_out())
coefs = best.named_steps["clf"].coef_
classes = best.named_steps["clf"].classes_

top = {}
for i, cls in enumerate(classes):
    order = coefs[i].argsort()
    top[f"{cls} ↑"] = feature_names[order[-15:][::-1]]
    top[f"{cls} ↓"] = feature_names[order[:15]]

pd.DataFrame(top)

In [ ]:
pred_full = best.predict(X_test)
cm = confusion_matrix(y_test, pred_full, labels=classes)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)), classes); ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel("предсказано"); ax.set_ylabel("истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("Матрица ошибок"); plt.tight_layout(); plt.show()

print(classification_report(y_test, pred_full, digits=3))

In [ ]:
# самые уверенные ошибки 
proba = best.predict_proba(X_test)
conf = proba.max(axis=1)
wrong = pred_full != y_test.values

errors = pd.DataFrame({
    "истина": y_test.values[wrong],
    "предсказано": pred_full[wrong],
    "уверенность": conf[wrong].round(3),
    "текст": X_test.values[wrong],
}).sort_values("уверенность", ascending=False).head(10)

errors.reset_index(drop=True)